# vLLM Multi-Gemma — Build & Push to GHCR

This notebook builds the Docker image and pushes it to GitHub Container Registry.
After that, you manage everything from vast.ai web UI / terminal.

**You need:** GitHub PAT (classic) with `write:packages` + `read:packages` scopes.

---

## 1. Configuration

In [ ]:
GITHUB_USER = "mrzduru"                          # your GitHub username
GITHUB_PAT  = "ghp_xxxxxxxxxxxxxxxxxxxx"          # PAT with write:packages

IMAGE_NAME = f"ghcr.io/{GITHUB_USER}/vllm-multi-gemma"
IMAGE_TAG  = "latest"
FULL_IMAGE = f"{IMAGE_NAME}:{IMAGE_TAG}"

print(f"Will build and push: {FULL_IMAGE}")

## 2. Clone Repo

In [ ]:
%%bash -s "$GITHUB_USER" "$GITHUB_PAT"

REPO_URL="https://$1:$2@github.com/$1/vllm-multi-gemma.git"

if [ -d "vllm-multi-gemma" ]; then
    cd vllm-multi-gemma && git pull
else
    git clone "$REPO_URL"
fi

echo "✓ Repo ready"

## 3. Build Docker Image

In [ ]:
%%bash -s "$FULL_IMAGE"

cd vllm-multi-gemma
echo "Building: $1"
docker build -t "$1" -f docker/Dockerfile .

echo ""
echo "✓ Build complete"
docker images | grep vllm-multi-gemma

## 4. Push to GHCR

In [ ]:
%%bash -s "$GITHUB_USER" "$GITHUB_PAT" "$FULL_IMAGE"

echo "$2" | docker login ghcr.io -u "$1" --password-stdin
echo "Pushing $3 ..."
docker push "$3"

echo ""
echo "✓ Pushed: $3"
echo ""
echo "Next step: make the package public at:"
echo "  https://github.com/users/$1/packages/container/vllm-multi-gemma/settings"
echo "  → Danger Zone → Change visibility → Public"
echo ""
echo "Then on vast.ai, use this image:"
echo "  $3"

## Done!

Your image is at `ghcr.io/mrzduru/vllm-multi-gemma:latest`.

### On vast.ai:

1. Rent a GPU instance (≥32GB VRAM)
2. Set the Docker image to `ghcr.io/mrzduru/vllm-multi-gemma:latest`
3. Set environment variables:
   - `MODEL_NAME=gemma2-cosmos`
   - `HF_TOKEN=hf_your_token`
   - `PORT=8000`
4. Expose port `8000`
5. Once running, your endpoint is `https://INSTANCE_ID-8000.proxy.vast.ai/v1`

### Switch models via SSH:
```bash
/opt/vllm-gemma/scripts/switch_model.sh gemma3-12b
/opt/vllm-gemma/scripts/switch_model.sh gemma4-12b
/opt/vllm-gemma/scripts/switch_model.sh gemma2-cosmos
```